# Chapter 10: Tool Engineering
## Topic 5: Error, Timeout, and Retry Handling

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Every tool demonstrated so far in this notebook series — `validate_fd_reference`, `search_knowledge_base` — has been shown working correctly, on the happy path. This topic exists to address what every real production system eventually faces: what happens when a tool's real underlying dependency (a database connection, a network call, a file read) genuinely fails, times out, or behaves unexpectedly, mid-conversation, with a model actively waiting on the result?
- The central distinction this topic establishes, directly building on Topic 3's found/not-found principle: **a tool returning `found: False` is not an error** — it's a normal, valid, expected outcome the tool was specifically designed to represent. A genuine *error* is something different in kind: the lookup mechanism itself couldn't complete at all — a database connection dropped, a network request timed out, a file was unreadable — meaning the tool has no reliable answer to give, positive or negative. Conflating these two categories (treating "not found" and "system failed" identically) throws away exactly the distinction Topic 3 argued was essential.
- Why this matters specifically for a model-facing tool, rather than an ordinary function call: when an ordinary function fails, the calling code can catch an exception and decide what to do. When a *tool* fails mid-agent-loop, the calling code additionally has to decide *what to tell the model* — because the model is actively waiting for a `tool_result` to continue reasoning, and simply crashing the whole request loses everything the model had already correctly figured out in earlier turns.


### 2. Internal Working — Step by Step

**Three genuinely different failure categories a tool call can encounter, and why each needs different handling:**

1. **A legitimate negative result** (Topic 3's `found: False`) — the underlying system worked correctly and gave a definitive, valid answer that happens to be "no." This isn't an error at all, and should never be handled by the error/retry machinery this topic covers — retrying a lookup that correctly and definitively found nothing would be pointless, since the answer won't change on a second attempt.
2. **A transient error** — a timeout, a temporary network blip, a momentarily unavailable database connection — where the underlying system's *true* answer is still unknown, and asking again, perhaps after a brief pause, has a real chance of succeeding. This is the category retry logic exists specifically for.
3. **A permanent or non-retryable error** — a malformed request the dispatch layer's validation should have caught before even reaching the real function (Topic 1's Module 3), a fundamentally broken dependency, or any failure where retrying with the exact same input will predictably fail again in exactly the same way. Retrying this category wastes time and calls without any real chance of a different outcome.

**The retry mechanism itself, step by step, for the transient-error category specifically:**

1. **Attempt the real tool call.** If it succeeds (even with a legitimate `found: False` result), return that result immediately — no retry needed, since this isn't an error at all.
2. **If a transient-error exception occurs, wait briefly, then attempt the call again**, up to some maximum number of attempts. Waiting between attempts (rather than immediately retrying) gives a genuinely transient problem — like a brief network hiccup — a real chance to resolve itself before trying again.
3. **If every retry attempt is exhausted without success, stop retrying and construct an explicit, honest error result** — not a crash, and not a fabricated "not found," but a distinct, clearly-labeled outcome stating that the tool could not complete its check at all.
4. **Feed this explicit error result back to the model, exactly as any other `tool_result`**, using the same message mechanics from Topic 2 — the model needs to see, in its own conversation history, that this attempt genuinely failed to produce an answer, so it can reason accordingly (for example, telling the customer there's a temporary system issue, rather than confidently stating something false based on a fabricated result).


### 3. How This Is Implemented in This Project

- `validate_fd_reference`'s current implementation, as shown in Topic 3, returns `found: False` for a genuinely nonexistent reference number — this project's existing design already correctly treats this as category 1 (a legitimate negative result), not an error, exactly matching this topic's first principle.
- Extending this tool with genuine error handling means wrapping the underlying `get_fd_record()` call (which, in a production system backed by a real database rather than this project's CSV file, could genuinely time out or fail to connect) in retry logic specifically for category 2 failures, while leaving the `found: False` path for category 1 completely untouched — these are structurally different code paths, not variations of the same one.
- The result fed back to the model after exhausting retries needs its own distinct shape, clearly different from both `found: True` and `found: False` — something like `{"error": "lookup_unavailable", "reference_number": reference_number}` — so the model (and any downstream faithfulness or citation checking, Chapter 8 Topic 3/4) can distinguish "the system told us definitively this doesn't exist" from "the system couldn't tell us anything at all right now," which call for genuinely different customer-facing responses.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Retrying a non-retryable error wastes time and calls without any chance of a different outcome — and can make user-facing latency meaningfully worse.** If a malformed reference number (which will fail identically every single time regardless of how many times it's retried) is treated as a transient error and retried several times with waits between attempts, the customer experiences significantly worse latency for a failure that was never going to succeed no matter how many attempts were made. This is precisely why Topic 1's Module 3 validation (catching malformed input) needs to happen *before* the retry-eligible code path, not be conflated with it.
- **Retry logic without a maximum attempt limit is a genuine, serious risk** — an agent loop already has `max_steps` bounding how many tool-call round trips a conversation can accumulate (Topic 2), but *within* a single tool call, unbounded retries against a persistently failing dependency could hang that single step indefinitely, defeating the outer loop's own bound entirely.
- **The wait between retry attempts (often implemented with an increasing delay between successive attempts) matters for not overwhelming an already-struggling dependency.** Retrying immediately and repeatedly against a database that's failing because it's overloaded can make the underlying problem worse, not better — spacing out attempts gives the dependency genuine breathing room to recover.
- **Monitoring:** logging every retry attempt (not just the final outcome) — including which attempt number succeeded, if any, and the total time spent retrying — creates the visibility needed to distinguish "this tool experiences occasional, self-resolving transient issues" from "this tool has a persistent, worsening reliability problem" that needs real, structural investigation rather than just more retries.
- **Cost and latency:** each retry attempt costs real time (and, for network-based tools, potentially real API/infrastructure cost) — the total worst-case latency of a tool call is not its single-attempt latency, but its single-attempt latency multiplied by the maximum retry count, plus the cumulative wait time between attempts. This worst case needs to be accounted for in any latency budget the surrounding agent loop or user-facing system design assumes.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **How many retry attempts to allow, and how long to wait between them:** more attempts and longer waits increase the chance a genuinely transient problem resolves itself before giving up, at the direct cost of worse worst-case latency for the customer waiting on a response. This is a real, explicit trade-off — not a "more retries is always better" situation — and should be tuned based on the actual, measured frequency and typical duration of transient failures for a specific dependency, not set arbitrarily.
- **What to tell the model (and ultimately the customer) after retries are exhausted:** an honest, explicit "the system could not complete this check right now" result respects the customer by not fabricating an answer, but might mean the customer's original question genuinely can't be answered in this interaction — versus a policy of escalating to human review specifically for this failure category, which costs more (human time) but avoids leaving the customer with no path forward. This is a product and domain decision layered on top of the core technical retry mechanism, not purely a technical one.
- **Whether retry logic belongs inside the tool function itself, or in a separate wrapper/decorator applied uniformly across multiple tools:** placing it inside each tool function individually gives fine-grained, tool-specific control (a lightweight lookup might need a different retry policy than a heavier external API call), while a shared wrapper reduces duplicated logic across many tools at the cost of less per-tool customization — this becomes a more pressing design question specifically once Topic 6's multi-tool agent introduces several tools that might each need broadly similar, but not necessarily identical, retry handling.


### 6. Alternatives and When to Use Each

- **No retry logic at all, fail immediately on any error:** appropriate only for a tool backed by a dependency with a genuinely negligible transient-failure rate, or for an early-stage prototype where this level of resilience isn't yet worth the implementation complexity — not appropriate for any tool backed by a network call or external database in a production system.
- **Fixed-delay retry (wait the same amount of time between every attempt):** simple to implement, reasonable for a dependency with brief, consistently-short transient issues.
- **Increasing-delay retry (wait longer between each successive attempt):** better suited to a dependency that might be temporarily overloaded, since it avoids repeatedly hammering a struggling system with immediate, back-to-back retries — generally the more robust default for network-based or shared-infrastructure dependencies.
- **No retry, immediate escalation to human review on any error:** appropriate for a genuinely high-stakes tool where even a brief, honest "couldn't complete this check" result to the customer is unacceptable, and the cost of immediate human involvement is justified by the stakes.


### 7. Common Mistakes and Production Failures

- Conflating a legitimate negative result (Topic 3's `found: False`) with a genuine error, applying retry logic to a case where retrying can never change the outcome and only adds pointless latency.
- Retrying a non-retryable error (like a malformed request that will fail identically every time) instead of catching it at the dispatch-validation layer (Topic 1) before it ever reaches retry-eligible code.
- Implementing retry logic with no maximum attempt limit, risking a single tool call hanging indefinitely against a persistently failing dependency.
- Retrying immediately and repeatedly with no delay between attempts, potentially worsening an already-overloaded dependency's problem rather than giving it room to recover.
- Not giving the model (and thus the final customer-facing answer) a distinct, honest signal when retries are exhausted, either crashing the whole request or silently fabricating a result instead of clearly communicating that the check genuinely could not be completed.
- Not accounting for the full worst-case latency (single-attempt time × maximum retries, plus wait time) when reasoning about a tool's impact on overall response latency.


### 8. Lead-Level Interview Questions

**Basic**

- Q: Why shouldn't a tool's legitimate `found: False` result be treated as an error requiring retry?
  A: `found: False` means the underlying system worked correctly and gave a definitive, valid answer — the reference number genuinely doesn't exist. Retrying wouldn't change this outcome, since nothing about the system's state was actually uncertain or incomplete; it was a successful lookup that happened to return a negative result, not a failure to complete the lookup at all.

- Q: What's the difference between a transient error and a permanent (non-retryable) error, and why does this distinction matter for retry logic?
  A: A transient error (like a brief network timeout) has a real chance of succeeding if attempted again, since the underlying problem may resolve on its own. A permanent error (like a fundamentally malformed request) will fail identically no matter how many times it's retried. Retrying a permanent error wastes time and calls with zero chance of a different outcome — this distinction is exactly why malformed-input validation (Topic 1) should happen before any retry-eligible code path, not be conflated with it.

**Intermediate**

- Q: What should happen when a tool's retry attempts are all exhausted without success?
  A: The tool should return a distinct, honest, explicitly-labeled error result — different in shape from both a `found: True` success and a `found: False` legitimate negative — communicating that the check could not be completed at all. This result gets fed back to the model exactly like any other tool result, so the model can reason accordingly (for example, informing the customer of a temporary system issue) rather than either crashing the whole request or fabricating a plausible-sounding but unfounded answer.

- Q: Why does waiting between retry attempts matter, rather than retrying immediately and repeatedly?
  A: Retrying immediately and repeatedly against a dependency that's failing because it's overloaded can make the underlying problem worse by adding more load to an already-struggling system. Waiting between attempts — especially with an increasing delay — gives a genuinely transient problem real room to resolve before the next attempt, improving the odds that a retry actually succeeds rather than just repeating the same failure faster.

**Advanced**

- Q: Design the full error-handling flow for a tool call, from the moment the underlying dependency fails to the moment the model sees a result, distinguishing all the relevant failure categories.
  A: First, dispatch-layer validation (Topic 1) checks the request's basic well-formedness before any real execution — a malformed request is rejected here, never reaching retry logic at all. If the request passes validation and the real function executes, a genuine transient error (like a timeout) triggers the retry mechanism: wait, attempt again, up to a maximum count, with increasing delay between attempts. If a retry succeeds, return that result normally. If all retries are exhausted, construct a distinct, explicitly-labeled error result (not a fabricated `found: False`, not a crash) and feed it back to the model as a `tool_result`, using the same message mechanics as any successful call (Topic 2) — the model must see this failure state explicitly in its own context to reason about it correctly.

- Q: A tool experiences a rising rate of exhausted-retry errors over time. How would you decide whether to adjust the retry policy (more attempts, longer waits) versus investigating the underlying dependency itself?
  A: Adjusting the retry policy treats the symptom, and is only appropriate if the underlying failures are genuinely brief and self-resolving, just needing slightly more patience to catch on a retry. A rising rate over time, rather than a stable, low background rate, is a signal the underlying dependency itself has a real, worsening reliability problem — more retries would only mask this, delaying detection while making the customer wait even longer for calls that are increasingly likely to fail regardless. The retry logging discussed in this topic (attempt counts, timing, ultimate outcomes) is exactly the evidence needed to distinguish which situation is actually occurring.

**Scenario-based**

- Q: In production, customers occasionally report unusually long waits for an FD status check, and some report the assistant said it "couldn't check the account status right now." Walk through your investigation.
  A: The explicit "couldn't check right now" message is the intended behavior for the exhausted-retries error case, not a bug in itself — the actual investigation should focus on how *often* this is happening and why. Check the retry logs for this tool specifically: is the underlying dependency failing more often than expected, and is the current retry count/delay policy appropriate given how long transient failures typically take to resolve, or is it retrying too few times too quickly, or too many times with too much cumulative delay before giving up? This log-driven evidence should directly inform whether the fix is tuning the retry policy or escalating an investigation into the underlying dependency's reliability itself.

**Follow-up questions to expect:**

- "How would you decide the right maximum retry count and delay for a specific tool's dependency?"
- "What would you log to distinguish a transient dependency issue from a persistent one?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **This topic's three-category failure distinction (legitimate negative result, transient error, permanent error) is a general software reliability engineering pattern, not unique to LLM tool-calling** — the same categorization and the same retry-with-backoff approach for transient failures specifically appears constantly in distributed systems and network programming generally, well beyond this specific application.
- **Retry logic and the dispatch-layer validation from Topic 1 form another instance of this notebook series' recurring defense-in-depth pattern** — validation prevents non-retryable errors from ever reaching retry-eligible code, and retry logic handles the genuinely transient failures validation was never meant to catch, each covering a distinct category the other doesn't address.
- **Feeding an honest error result back to the model, rather than crashing or fabricating success, is a specific instance of the broader "fail safe, not fail silent" principle** already established for citation verification (Chapter 8 Topic 2) and metadata-filtering access control (Chapter 7 Topic 8) — recognizing this recurring principle across multiple, seemingly-different topics is a mark of genuinely transferable understanding.

### 10. Quick Revision Summary (for last-minute interview prep)

> A tool call can fail in three genuinely different ways, each requiring different handling: a legitimate negative result (like `found: False`) is not an error at all and should never trigger retry logic; a transient error (a timeout, a brief network issue) has a real chance of succeeding if attempted again and is exactly what retry logic exists for; and a permanent, non-retryable error (like a malformed request) will fail identically no matter how many times it's retried, and should be caught by dispatch-layer validation (Topic 1) before ever reaching retry-eligible code. Retry logic itself needs a maximum attempt count (to avoid hanging indefinitely) and a delay between attempts (ideally increasing, to avoid overwhelming an already-struggling dependency and to give genuinely transient issues real room to resolve). When retries are exhausted, the tool must return a distinct, explicitly-labeled error result — never a fabricated success or negative result, and never a silent crash — fed back to the model using the same `tool_result` message mechanics (Topic 2) as any successful call, so the model can reason honestly about a genuine failure to complete the check. The full worst-case latency of a tool call (single-attempt time × max retries, plus cumulative wait time) needs to be accounted for in any surrounding latency budget, and rising retry-exhaustion rates over time are a signal to investigate the underlying dependency's reliability, not just to add more retries.


### Module 1: Three Failure Categories, Modeled Distinctly

Build a simulated dependency that can produce all three failure categories (legitimate negative, transient error, permanent error), and show why each needs different handling.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Three failure categories, modeled distinctly
# ------------------------------------------------------------------

import time

class TransientError(Exception):
    """A real, distinguishable exception type for transient failures --
    e.g. a timeout or brief network issue. Retrying THIS category has
    a real chance of succeeding."""
    pass

class PermanentError(Exception):
    """A real, distinguishable exception type for non-retryable
    failures -- e.g. malformed input. Retrying THIS category is
    pointless; it will fail identically every time."""
    pass


FD_RECORDS_TABLE = {
    "BJ2019FD7717": {"Customer_Name": "Shobha Chopra", "Status": "Closed (Premature)"},
}

# a controllable simulated dependency: fails a specified number of
# times with a TransientError, then succeeds -- lets us test retry
# logic against a REAL, controllable failure pattern
class SimulatedFlakyDependency:
    def __init__(self, fail_count: int, permanent_failure: bool = False):
        self.fail_count = fail_count
        self.permanent_failure = permanent_failure
        self.attempts_made = 0

    def query(self, reference_number: str) -> dict:
        self.attempts_made += 1
        if self.permanent_failure:
            raise PermanentError(f"Malformed query for '{reference_number}' -- will fail every time.")
        if self.attempts_made <= self.fail_count:
            raise TransientError(f"Attempt {self.attempts_made}: connection timed out.")
        # success -- either a real record or a legitimate not-found
        record = FD_RECORDS_TABLE.get(reference_number)
        if record is None:
            return {"found": False, "reference_number": reference_number}
        return {"found": True, "reference_number": reference_number, **record}


print("=" * 70)
print("MODULE 1: THREE FAILURE CATEGORIES")
print("=" * 70)

# Category 1: legitimate negative result -- NOT an error at all
reliable_dep = SimulatedFlakyDependency(fail_count=0)
legit_negative = reliable_dep.query("BJ9999FD0000")
print(f"\n[Category 1: legitimate negative result]")
print(f"  Result: {legit_negative}")
print(f"  This is NOT an error -- no retry needed, the answer is definitively 'no'.")

# Category 2: transient error -- demonstrated by raising, then catching
transient_dep = SimulatedFlakyDependency(fail_count=2)
print(f"\n[Category 2: transient error -- will fail twice, then succeed]")
try:
    transient_dep.query("BJ2019FD7717")
except TransientError as e:
    print(f"  Attempt 1 raised: {e}")

# Category 3: permanent error -- will NEVER succeed no matter how many attempts
permanent_dep = SimulatedFlakyDependency(fail_count=0, permanent_failure=True)
print(f"\n[Category 3: permanent error -- will fail EVERY attempt, forever]")
try:
    permanent_dep.query("malformed-input")
except PermanentError as e:
    print(f"  Attempt raised: {e}")
    print(f"  Retrying this would be POINTLESS -- it will fail identically every time.")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: THREE FAILURE CATEGORIES

[Category 1: legitimate negative result]
  Result: {'found': False, 'reference_number': 'BJ9999FD0000'}
  This is NOT an error -- no retry needed, the answer is definitively 'no'.

[Category 2: transient error -- will fail twice, then succeed]
  Attempt 1 raised: Attempt 1: connection timed out.

[Category 3: permanent error -- will fail EVERY attempt, forever]
  Attempt raised: Malformed query for 'malformed-input' -- will fail every time.
  Retrying this would be POINTLESS -- it will fail identically every time.

Module 1 complete. Run Module 2 next.


### Module 2: Real Retry-With-Backoff Logic, Measured Against the Simulated Transient Failure

Implement actual retry logic with increasing delay, and measure the REAL time it takes to succeed against a dependency that fails a controlled number of times first.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Real retry-with-backoff, measured against real timing
# ------------------------------------------------------------------

def call_with_retry(query_func, reference_number: str, max_attempts: int = 4,
                     base_delay_seconds: float = 0.2) -> dict:
    """REAL retry-with-increasing-delay logic. Only retries on
    TransientError -- a PermanentError is NOT retried at all, since
    retrying it can never succeed (Category 3 from Module 1)."""
    last_exception = None
    for attempt in range(1, max_attempts + 1):
        try:
            result = query_func(reference_number)
            return {"status": "success", "result": result, "attempts_used": attempt}
        except TransientError as e:
            last_exception = e
            if attempt < max_attempts:
                delay = base_delay_seconds * attempt  # INCREASING delay each attempt
                time.sleep(delay)
        except PermanentError as e:
            # NOT retried -- fail immediately, exactly per Category 3's principle
            return {"status": "permanent_failure", "error": str(e), "attempts_used": attempt}

    return {"status": "retries_exhausted", "error": str(last_exception), "attempts_used": max_attempts}


print("=" * 70)
print("REAL RETRY-WITH-BACKOFF, MEASURED TIMING")
print("=" * 70)

# Case A: transient failure that RESOLVES within the retry budget
dep_resolves = SimulatedFlakyDependency(fail_count=2)
start = time.perf_counter()
result_a = call_with_retry(dep_resolves.query, "BJ2019FD7717", max_attempts=4)
elapsed_a = time.perf_counter() - start

print(f"\n[Case A: fails twice, succeeds on 3rd attempt]")
result_a_status = result_a["status"]
print(f"  Status: {result_a_status}")
result_a_attempts_used = result_a["attempts_used"]
print(f"  Attempts used: {result_a_attempts_used}")
print(f"  Real elapsed time: {elapsed_a:.3f}s (includes REAL backoff delays)")
if result_a["status"] == "success":
    result_a_result = result_a["result"]
    print(f"  Result: {result_a_result}")

# Case B: transient failure that NEVER resolves within the retry budget
dep_never_resolves = SimulatedFlakyDependency(fail_count=10)  # fails more than max_attempts
start = time.perf_counter()
result_b = call_with_retry(dep_never_resolves.query, "BJ2019FD7717", max_attempts=4)
elapsed_b = time.perf_counter() - start

print(f"\n[Case B: fails 10 times -- exceeds retry budget of 4 attempts]")
result_b_status = result_b["status"]
print(f"  Status: {result_b_status}")
result_b_attempts_used = result_b["attempts_used"]
print(f"  Attempts used: {result_b_attempts_used}")
print(f"  Real elapsed time: {elapsed_b:.3f}s (full retry budget consumed)")

# Case C: PERMANENT error -- should NOT retry at all, should fail fast
dep_permanent = SimulatedFlakyDependency(fail_count=0, permanent_failure=True)
start = time.perf_counter()
result_c = call_with_retry(dep_permanent.query, "malformed-input", max_attempts=4)
elapsed_c = time.perf_counter() - start

print(f"\n[Case C: PERMANENT error -- should fail FAST, no wasted retries]")
result_c_status = result_c["status"]
print(f"  Status: {result_c_status}")
result_c_attempts_used = result_c["attempts_used"]
print(f"  Attempts used: {result_c_attempts_used} (should be 1, NOT 4)")
print(f"  Real elapsed time: {elapsed_c:.3f}s (should be near-instant, no backoff delays)")

if result_c["attempts_used"] == 1 and elapsed_c < elapsed_b:
    print(f"\nCONFIRMED: the permanent error failed on attempt 1 with near-zero delay,")
    print(f"while the exhausted-transient-retry case (B) consumed the FULL retry")
    print(f"budget and backoff time ({elapsed_b:.3f}s) -- exactly the theory's claim")
    print(f"that conflating these categories would waste real, measurable time.")

print("\nModule 2 complete. Run Module 3 next.")


REAL RETRY-WITH-BACKOFF, MEASURED TIMING

[Case A: fails twice, succeeds on 3rd attempt]
  Status: success
  Attempts used: 3
  Real elapsed time: 0.600s (includes REAL backoff delays)
  Result: {'found': True, 'reference_number': 'BJ2019FD7717', 'Customer_Name': 'Shobha Chopra', 'Status': 'Closed (Premature)'}

[Case B: fails 10 times -- exceeds retry budget of 4 attempts]
  Status: retries_exhausted
  Attempts used: 4
  Real elapsed time: 1.200s (full retry budget consumed)

[Case C: PERMANENT error -- should fail FAST, no wasted retries]
  Status: permanent_failure
  Attempts used: 1 (should be 1, NOT 4)
  Real elapsed time: 0.000s (should be near-instant, no backoff delays)

CONFIRMED: the permanent error failed on attempt 1 with near-zero delay,
while the exhausted-transient-retry case (B) consumed the FULL retry
budget and backoff time (1.200s) -- exactly the theory's claim
that conflating these categories would waste real, measurable time.

Module 2 complete. Run Module 3 next.


### Module 3: The Distinct Error Result Fed Back to the Model

Shows the three genuinely different result shapes (success, legitimate not-found, exhausted-retry error) and how each would be packaged as a tool_result using Topic 2's real message mechanics.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Distinct result shapes fed back to the model
# ------------------------------------------------------------------

import json

def build_tool_result_block(retry_outcome: dict, tool_use_id: str) -> dict:
    """Builds the ACTUAL tool_result block that would be sent back to
    the model, using Topic 2's real message mechanics -- but shaping
    the CONTENT differently depending on which of the three outcomes
    actually occurred, so the model can reason about them differently."""
    if retry_outcome["status"] == "success":
        content = retry_outcome["result"]  # either found=True or found=False -- BOTH legitimate
    elif retry_outcome["status"] == "permanent_failure":
        content = {"error": "invalid_request", "detail": retry_outcome["error"]}
    else:  # retries_exhausted
        content = {"error": "lookup_unavailable",
                   "detail": "Could not complete this check after multiple attempts. Please inform the customer of a temporary system issue."}

    return {"type": "tool_result", "tool_use_id": tool_use_id, "content": json.dumps(content)}


print("=" * 70)
print("THREE DISTINCT tool_result SHAPES SENT BACK TO THE MODEL")
print("=" * 70)

# reuse results from Module 2
scenarios = [
    ("Legitimate found=True (success)", {"status": "success", "result": {"found": True, "reference_number": "BJ2019FD7717", "Customer_Name": "Shobha Chopra"}}),
    ("Legitimate found=False (success, negative)", {"status": "success", "result": {"found": False, "reference_number": "BJ9999FD0000"}}),
    ("Retries exhausted (genuine system failure)", result_b),
    ("Permanent/malformed error (never retried)", result_c),
]

for label, outcome in scenarios:
    tool_result = build_tool_result_block(outcome, tool_use_id="toolu_test")
    print(f"\n[{label}]")
    tool_result_content = tool_result["content"]
    print(f"  tool_result sent to model: {tool_result_content}")

print("\nNotice all FOUR scenarios produce a DIFFERENT, explicit, honest")
print("content shape -- the model can distinguish a genuine record, a")
print("legitimate not-found, an unavailable-system state, and an invalid")
print("request from each other, and reason (and respond to the customer)")
print("appropriately for each, rather than these being silently collapsed")
print("into one ambiguous 'something went wrong' signal.")

print("\nModule 3 complete. All Chapter 10 Topic 5 modules done.")
print()
print("=" * 70)
print("CHAPTER 10 TOPIC 5 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  THREE distinct failure categories, each needing DIFFERENT handling:
  legitimate negative result (never retry), transient error (retry with
  backoff), permanent error (never retry, fail fast).

  Retry logic needs a MAXIMUM attempt count and INCREASING delay
  between attempts -- demonstrated with real measured timing showing
  the exhausted-retry case consumes real, bounded time.

  Permanent errors must fail FAST, with zero wasted retry attempts --
  demonstrated concretely: 1 attempt, near-zero elapsed time, vs the
  full retry budget consumed for a genuinely transient, unresolved
  failure.

  Every outcome (success/found=True, success/found=False, exhausted-
  retries error, permanent error) needs its OWN distinct, honest result
  shape fed back to the model -- never collapse them into one signal.
""")


THREE DISTINCT tool_result SHAPES SENT BACK TO THE MODEL

[Legitimate found=True (success)]
  tool_result sent to model: {"found": true, "reference_number": "BJ2019FD7717", "Customer_Name": "Shobha Chopra"}

[Legitimate found=False (success, negative)]
  tool_result sent to model: {"found": false, "reference_number": "BJ9999FD0000"}

[Retries exhausted (genuine system failure)]
  tool_result sent to model: {"error": "lookup_unavailable", "detail": "Could not complete this check after multiple attempts. Please inform the customer of a temporary system issue."}

[Permanent/malformed error (never retried)]
  tool_result sent to model: {"error": "invalid_request", "detail": "Malformed query for 'malformed-input' -- will fail every time."}

Notice all FOUR scenarios produce a DIFFERENT, explicit, honest
content shape -- the model can distinguish a genuine record, a
legitimate not-found, an unavailable-system state, and an invalid
request from each other, and reason (and respond to the custo